# 03 — Astrometric Quality

**Plots:** PM uncertainty vs magnitude · improvement factor · parallax uncertainty · position offset · N_HST histogram · correlation distributions

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
# bp3m is installed as a package — no path manipulation needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results,
    gaia_pm_sigma, bp3m_pm_sigma,
    gaia_pos_sigma, bp3m_pos_sigma,
    bp3m_full_cov,
)

DATA = Path(OUTPUT_DIR)
_gaia_files = sorted((DATA / FIELD_NAME / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])
bp3m = load_bp3m_results(DATA / FIELD_NAME / 'BP3M_results')
stars = bp3m['stars']

sig_pm_gaia  = gaia_pm_sigma(gaia)
sig_pm_bp3m  = bp3m_pm_sigma(bp3m)
sig_pos_gaia = gaia_pos_sigma(gaia)
sig_pos_bp3m = bp3m_pos_sigma(bp3m)

gmag = stars['gmag'].values
print(f'Stars: {len(stars)}')

In [ ]:
# ── PM uncertainty vs magnitude ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

n_hst = stars['n_hst_used'].values

for ax, sig, title, color in [
    (axes[0], sig_pm_gaia, 'Gaia DR3', 'steelblue'),
    (axes[1], sig_pm_bp3m, 'BP3M (this work)', 'darkorange'),
]:
    ok = np.isfinite(gmag) & np.isfinite(sig) & (sig > 0)
    if ax is axes[1]:
        sc = ax.scatter(gmag[ok], sig[ok], c=n_hst[ok], cmap='plasma',
                        s=4, alpha=0.5, rasterized=True)
        plt.colorbar(sc, ax=ax, label='N HST detections')
    else:
        ax.scatter(gmag[ok], sig[ok], s=4, alpha=0.5, c=color, rasterized=True)
    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel(r'$\sigma_{\mu}$ (mas yr$^{-1}$)')
    ax.set_yscale('log')
    ax.set_title(title)

fig.suptitle(f'{FIELD_NAME} — PM uncertainty vs magnitude', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_pm_sigma_vs_mag.png', dpi=150)
plt.show()

In [ ]:
# ── PM improvement factor ─────────────────────────────────────────────────────
ok = np.isfinite(gmag) & np.isfinite(sig_pm_gaia) & np.isfinite(sig_pm_bp3m)
ok &= (sig_pm_bp3m > 0) & (sig_pm_gaia > 0)
improvement = sig_pm_gaia[ok] / sig_pm_bp3m[ok]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(gmag[ok], improvement, c=n_hst[ok], cmap='plasma',
                s=4, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='N HST detections')
ax.axhline(1.0, color='k', lw=1, ls='--', label='No improvement')
ax.set_xlabel('Gaia G (mag)')
ax.set_ylabel(r'$\sigma_{\mu,\,\mathrm{Gaia}} \;/\; \sigma_{\mu,\,\mathrm{BP3M}}$')
ax.set_title(f'{FIELD_NAME} — PM improvement factor')
ax.legend()
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_pm_improvement.png', dpi=150)
plt.show()

print(f'Median improvement: {np.median(improvement):.2f}×')
print(f'Stars with >2× improvement: {(improvement > 2).sum()} / {ok.sum()}')

In [ ]:
# ── Parallax uncertainty vs magnitude ────────────────────────────────────────
from bp3m.pipeline.explore_utils import build_gaia_cov
C_gaia = build_gaia_cov(gaia)
C_bp3m = bp3m_full_cov(bp3m)

sig_plx_gaia = np.sqrt(C_gaia[:, 4, 4])
sig_plx_bp3m = np.sqrt(C_bp3m[:, 4, 4])

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, sig, title, color in [
    (axes[0], sig_plx_gaia, 'Gaia DR3', 'steelblue'),
    (axes[1], sig_plx_bp3m, 'BP3M (this work)', 'darkorange'),
]:
    ok = np.isfinite(gmag) & np.isfinite(sig) & (sig > 0)
    ax.scatter(gmag[ok], sig[ok], s=4, alpha=0.5, c=color, rasterized=True)
    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel(r'$\sigma_{\varpi}$ (mas)')
    ax.set_yscale('log')
    ax.set_title(title)

fig.suptitle(f'{FIELD_NAME} — Parallax uncertainty vs magnitude', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_plx_sigma_vs_mag.png', dpi=150)
plt.show()

In [ ]:
# ── Position offset (BP3M vs Gaia) ────────────────────────────────────────────
dra  = stars['delta_racosdec_bp3m'].values   # mas
ddec = stars['delta_dec_bp3m'].values        # mas

ok = np.isfinite(dra) & np.isfinite(ddec)
lim = 4 * 1.4826 * np.nanmedian(np.abs(dra[ok]))

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(dra[ok], ddec[ok], c=gmag[ok], cmap='viridis_r',
                s=4, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='Gaia G (mag)')
ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.5)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(r'$\Delta(\alpha\cos\delta)$ BP3M$-$Gaia (mas)')
ax.set_ylabel(r'$\Delta\delta$ BP3M$-$Gaia (mas)')
ax.set_title(f'{FIELD_NAME} — Position offset')
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_position_offset.png', dpi=150)
plt.show()

In [ ]:
# ── N_HST used per star ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.arange(0, n_hst.max() + 2) - 0.5
ax.hist(n_hst, bins=bins, color='steelblue', edgecolor='white', lw=0.5)
ax.set_xlabel('N HST detections used')
ax.set_ylabel('N stars')
ax.set_title(f'{FIELD_NAME} — HST detections per star')
ax.axvline(np.median(n_hst), color='darkorange', lw=1.5,
           label=f'Median = {np.median(n_hst):.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_n_hst_hist.png', dpi=150)
plt.show()

In [ ]:
# ── Correlation coefficient distributions ────────────────────────────────────
corr_cols = [
    ('corr_pmra_pmdec', r'$\rho(\mu_{\alpha^*},\,\mu_\delta)$'),
    ('corr_pmra_plx',   r'$\rho(\mu_{\alpha^*},\,\varpi)$'),
    ('corr_pmdec_plx',  r'$\rho(\mu_\delta,\,\varpi)$'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label) in zip(axes, corr_cols):
    if col not in stars.columns:
        ax.text(0.5, 0.5, f'{col}\nnot found', transform=ax.transAxes, ha='center')
        continue
    vals = stars[col].dropna().values
    ax.hist(vals, bins=40, color='darkorange', edgecolor='white', lw=0.5)
    ax.axvline(0, color='k', lw=1, ls='--', alpha=0.5)
    ax.set_xlabel(label)
    ax.set_ylabel('N stars')
    ax.set_title(f'Median = {np.nanmedian(vals):.3f}')

fig.suptitle(f'{FIELD_NAME} — BP3M posterior correlations', fontsize=13)
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_03_correlations.png', dpi=150)
plt.show()